In [33]:
import numpy as np
import pandas as pd
import random
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import HillClimbSearch, MaximumLikelihoodEstimator, BayesianEstimator
from imblearn.over_sampling import SMOTEN
from sklearn.utils import resample
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

In [27]:
# load data
DATA_PATH = "data/nsw_road_crash_data_2018-2022_crash.csv"

df = pd.read_csv(DATA_PATH, encoding="latin1")

In [ ]:
TARGET = "Degree of crash - detailed"

FEATURES = [
    "Day of week of crash",
    "Distance",
    "Identifying feature type",
    "Type of location",
    "Urbanisation",
    "Alignment",
    "Street lighting",
    "Road surface",
    "Surface condition",
    "Weather",
    "Natural lighting",
    "Signals operation",
    "Other traffic control",
    "Speed limit",
    "Road classification (admin)",
    "First impact type",
    "Key TU type",
    "Other TU type",
    "No. of traffic units involved",
]

data = df[FEATURES + [TARGET]].copy()

In [29]:
# remove rows with unknown values
def remove_unknown_strings_only(df):
    """
    Removes rows where any of the 20 columns contain the word 'unknown',
    but preserves rows that contain actual NaN/Null values.
    """
    # The 20 variables identified in the paper
    features = FEATURES
    
    # Create a mask for 'unknown'
    # We cast to string to use .str.contains, but we must handle NaNs 
    # so they don't get caught in the 'unknown' filter.
    contains_unknown = df[features].apply(
        lambda col: col.astype(str).str.contains('unknown', case=False, na=False)
    ).any(axis=1)
    
    # Filter the dataframe
    df_no_unknowns = df[~contains_unknown].copy()
    
    # Validation
    unknown_count = contains_unknown.sum()
    null_count = df_no_unknowns[features].isnull().any(axis=1).sum()
    
    print(f"Rows removed due to 'unknown' string: {unknown_count}")
    print(f"Rows remaining that still contain NaNs: {null_count}")
    print(f"New dataset shape: {df_no_unknowns.shape}")
    
    return df_no_unknowns

df = remove_unknown_strings_only(df)
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")

Rows removed due to 'unknown' string: 39076
Rows remaining that still contain NaNs: 20228
New dataset shape: (56797, 50)

Target distribution:
Degree of crash - detailed
Moderate Injury           20224
Serious Injury            17979
Non-casualty (towaway)     9657
Minor/Other Injury         7528
Fatal                      1409
Name: count, dtype: int64


In [30]:
# 2. Discretization (BNs require discrete variables)
# Binning Distance
data['Distance'] = pd.cut(data['Distance'], 
                          bins=[-1, 0, 10, 100, 1000, np.inf], 
                          labels=['0', '1-10', '11-100', '101-1000', '>1000'])

# Binning No. of traffic units involved
data['No. of traffic units involved'] = pd.cut(data['No. of traffic units involved'],
                                               bins=[0, 1, 2, 3, np.inf],
                                               labels=['1', '2', '3', '>3'])

# Ensure all columns are strings for the estimator
data = data.astype(str)

In [31]:
X = data[FEATURES]
y = data[TARGET]

# test/train split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=440, stratify=y
)
print(f"\nTrain size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

train_df = pd.concat([X_train, y_train], axis = 1).astype(str)


Train size: 67111, Test size: 28762


In [35]:
def balance(X_train, y_train):
    sampler = SMOTEN(random_state=440)
    X_res, y_res = sampler.fit_resample(X_train.astype(str), y_train)
    balanced_df = pd.concat([X_res, y_res], axis=1)


    return balanced_df

balanced_df = balance(X_train, y_train)

In [ ]:
def train_ensemble_bn(X_train, y_train, balanced_df, n_estimators=5, max_features=10):
    ensemble = []
    
    # 1. Balance the initial training set
    #sampler = SMOTEN(random_state=440)
    #X_res, y_res = sampler.fit_resample(X_train.astype(str), y_train)
    # balanced_df = pd.concat([X_res, y_res], axis=1)

    all_features = list(X_train.columns)
    target_col = y_train.name
    
    print(f"Starting Ensemble Training ({n_estimators} models)...")
    
    for i in range(n_estimators):
        print(f"  Training Model {i+1}/{n_estimators}...")
        
        # 2. Pick a random subset of features for this specific model
        # Choosing roughly sqrt(n) or 60-70% of features is standard
        selected_features = random.sample(all_features, k=min(max_features, len(all_features)))
        subset_df = balanced_df[selected_features + [target_col]]
        
        # 2. Create a bootstrap sample (with replacement) from the balanced data
        boot_df = resample(subset_df, replace=True, n_samples=len(subset_df), random_state=i)
        
        # 3. Learn Structure
        hc = HillClimbSearch(boot_df)
        # Using a small max_indegree to keep the ensemble fast
        best_structure = hc.estimate(scoring_method="bic-d", max_indegree=2)
        
        # 4. Fit Parameters
        model = DiscreteBayesianNetwork(best_structure.edges())
        # model.fit(boot_df, estimator=MaximumLikelihoodEstimator)
        model.fit(boot_df, estimator=BayesianEstimator, prior_type = "BDeu", equivalent_sample_size=10)
        
        ensemble.append((model, selected_features))
        
    return ensemble

# --- Execution ---
# Train the ensemble
bn_ensemble = train_ensemble_bn(X_train, y_train, balanced_df, n_estimators=5)


C:\Users\victo\AppData\Local\Temp\ipykernel_28280\3976260659.py:26: FutureWarning: HillClimbSearch is deprecated. Please use pgmpy.causal_discovery.HillClimbSearch instead.
  hc = HillClimbSearch(boot_df)


Starting Ensemble Training (5 models)...
  Training Model 1/5...


  0%|          | 18/1000000 [00:00<15:10:33, 18.30it/s]
C:\Users\victo\AppData\Local\Temp\ipykernel_28280\3976260659.py:26: FutureWarning: HillClimbSearch is deprecated. Please use pgmpy.causal_discovery.HillClimbSearch instead.
  hc = HillClimbSearch(boot_df)


  Training Model 2/5...


  0%|          | 19/1000000 [00:00<13:26:55, 20.65it/s]
C:\Users\victo\AppData\Local\Temp\ipykernel_28280\3976260659.py:26: FutureWarning: HillClimbSearch is deprecated. Please use pgmpy.causal_discovery.HillClimbSearch instead.
  hc = HillClimbSearch(boot_df)


...

  0%|          | 15/1000000 [00:01<18:55:34, 14.68it/s]
C:\Users\victo\AppData\Local\Temp\ipykernel_28280\3976260659.py:26: FutureWarning: HillClimbSearch is deprecated. Please use pgmpy.causal_discovery.HillClimbSearch instead.
  hc = HillClimbSearch(boot_df)


  Training Model 4/5...


  0%|          | 19/1000000 [00:00<13:41:50, 20.28it/s]
C:\Users\victo\AppData\Local\Temp\ipykernel_28280\3976260659.py:26: FutureWarning: HillClimbSearch is deprecated. Please use pgmpy.causal_discovery.HillClimbSearch instead.
  hc = HillClimbSearch(boot_df)


  Training Model 5/5...


  0%|          | 19/1000000 [00:00<13:04:16, 21.25it/s]


In [ ]:
from collections import Counter
import numpy as np

def predict_ensemble(ensemble, X_test, target_col):
    all_predictions = []
    
    #for i, model in enumerate(ensemble):
    for i, (model, features) in enumerate(ensemble):
        print(f"  Querying Model {i+1}...")
        
        X_test_subset = X_test[features].astype(str)
        
        # Handle unseen states for this specific model
        for col in features:
            valid_states = model.get_cpds(col).state_names[col]
            X_test_subset.loc[~X_test_subset[col].isin(valid_states), col] = None
            
        preds = model.predict(X_test_subset)
        all_predictions.append(preds[target_col].values)
        
    # 5. Aggregate results via Majority Vote (String-Safe)
    stacked_preds = np.array(all_predictions)
    
    print("  Aggregating votes...")
    # For each sample (column), find the most common string prediction
    final_preds = []

    final_preds = [Counter(stacked_preds[:, j]).most_common(1)[0][0] for j in range(stacked_preds.shape[1])]
    
    return np.array(final_preds)

In [38]:
# Predict
print("Generating ensemble predictions...")
y_pred_ensemble = predict_ensemble(bn_ensemble, X_test, y_train.name)

# Evaluate
print("\n--- Bagged Bayesian Network Results ---")
print(classification_report(y_test, y_pred_ensemble))

Generating ensemble predictions...
  Querying Model 1...


c:\Users\victo\AppData\Local\Programs\Python\Python313\Lib\site-packages\pgmpy\models\DiscreteBayesianNetwork.py:851: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data_unique_indexes = data.groupby(list(data.columns), dropna=False).apply(lambda t: t.index.tolist())
100%|██████████| 14513/14513 [00:56<00:00, 255.16it/s]


  Querying Model 2...


c:\Users\victo\AppData\Local\Programs\Python\Python313\Lib\site-packages\pgmpy\models\DiscreteBayesianNetwork.py:851: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data_unique_indexes = data.groupby(list(data.columns), dropna=False).apply(lambda t: t.index.tolist())
100%|██████████| 15447/15447 [01:21<00:00, 190.38it/s]


  Querying Model 3...


...

  Querying Model 4...


c:\Users\victo\AppData\Local\Programs\Python\Python313\Lib\site-packages\pgmpy\models\DiscreteBayesianNetwork.py:851: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data_unique_indexes = data.groupby(list(data.columns), dropna=False).apply(lambda t: t.index.tolist())
100%|██████████| 16510/16510 [01:24<00:00, 196.31it/s]


  Querying Model 5...


c:\Users\victo\AppData\Local\Programs\Python\Python313\Lib\site-packages\pgmpy\models\DiscreteBayesianNetwork.py:851: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  data_unique_indexes = data.groupby(list(data.columns), dropna=False).apply(lambda t: t.index.tolist())
100%|██████████| 20675/20675 [01:41<00:00, 203.99it/s]


  Aggregating votes...

--- Bagged Bayesian Network Results ---
                        precision    recall  f1-score   support

                 Fatal       0.02      0.17      0.03       433
    Minor/Other Injury       0.18      0.27      0.22      5166
       Moderate Injury       0.28      0.10      0.15      8022
Non-casualty (towaway)       0.32      0.35      0.34      9231
        Serious Injury       0.22      0.13      0.16      5910

              accuracy                           0.22     28762
             macro avg       0.20      0.20      0.18     28762
          weighted avg       0.26      0.22      0.22     28762



In [39]:
for i, (m, features) in enumerate(bn_ensemble):
    print(f"Model {i} edges: {len(m.edges())} | Features: {len(features)}")
    # This correctly unpacks the tuple so 'm' is the model object

Model 0 edges: 18 | Features: 10
Model 1 edges: 19 | Features: 10
Model 2 edges: 15 | Features: 10
Model 3 edges: 19 | Features: 10
Model 4 edges: 19 | Features: 10


In [40]:
for i, (model, features) in enumerate(bn_ensemble):
    print(f"\n--- Model {i+1} Structure ({len(model.edges())} edges) ---")
    print(f"Features available to this model: {features}")
    
    # Sort the edges to make it easier to read
    for u, v in sorted(model.edges()):
        print(f"  {u} ---> {v}")


--- Model 1 Structure (18 edges) ---
Features available to this model: ['Street lighting', 'Key TU type', 'Weather', 'Speed limit', 'Signals operation', 'Type of location', 'Other TU type', 'Road surface', 'Distance', 'No. of traffic units involved']
  Degree of crash - detailed ---> Key TU type
  Degree of crash - detailed ---> Other TU type
  Degree of crash - detailed ---> Speed limit
  Degree of crash - detailed ---> Street lighting
  Degree of crash - detailed ---> Weather
  Distance ---> Other TU type
  Distance ---> Type of location
  No. of traffic units involved ---> Key TU type
  No. of traffic units involved ---> Road surface
  Other TU type ---> No. of traffic units involved
  Speed limit ---> Distance
  Speed limit ---> Signals operation
  Speed limit ---> Type of location
  Street lighting ---> Distance
  Street lighting ---> Road surface
  Street lighting ---> Speed limit
  Street lighting ---> Weather
  Type of location ---> Signals operation

--- Model 2 Structure (19

In [41]:
from pgmpy.inference import VariableElimination
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize
from tqdm import tqdm
import numpy as np
import pandas as pd

def get_ensemble_probabilities(ensemble, X_test, target_col):
    n_samples = len(X_test)
    all_model_probs = []
    
    # Get consistent class labels from the first model
    first_model = ensemble[0][0]
    target_classes = sorted(first_model.get_cpds(target_col).state_names[target_col])
    num_classes = len(target_classes)
    
    for i, (model, features) in enumerate(ensemble):
        print(f"Inference: Model {i+1}/{len(ensemble)}")
        infer = VariableElimination(model)
        model_probs = np.zeros((n_samples, num_classes))
        
        for idx in tqdm(range(n_samples), desc="Processing rows"):
            row = X_test.iloc[idx]
            evidence = {}
            
            for col in features:
                if col in model.nodes():
                    val = str(row[col])
                    if val in model.get_cpds(col).state_names[col]:
                        evidence[col] = val
            
            try:
                result = infer.query(variables=[target_col], evidence=evidence, joint=False)
                prob_dist = result[target_col].values
                
                # Check for NaNs (the cause of your 0.50 score)
                if np.isnan(prob_dist).any():
                    model_probs[idx, :] = 1.0 / num_classes
                else:
                    model_probs[idx, :] = prob_dist
            except:
                model_probs[idx, :] = 1.0 / num_classes
        
        all_model_probs.append(model_probs)
        
    # Average the probabilities
    ensemble_probs = np.mean(all_model_probs, axis=0)
    return ensemble_probs, target_classes

# --- Run the Evaluation ---
target_col_name = "Degree of crash - detailed" 
y_prob_ensemble, classes = get_ensemble_probabilities(bn_ensemble, X_test, target_col_name)

# Ensure no NaNs leaked into the final ensemble
y_prob_ensemble = np.nan_to_num(y_prob_ensemble, nan=1.0/len(classes))

y_test_binarized = label_binarize(y_test, classes=classes)
total_auc = roc_auc_score(y_test_binarized, y_prob_ensemble, multi_class='ovr', average='weighted')

print(f"\n--- Improved Results ---")
print(f"Ensemble Graphical Model AUC-ROC: {total_auc:.4f}")
for i, class_name in enumerate(classes):
    class_auc = roc_auc_score(y_test_binarized[:, i], y_prob_ensemble[:, i])
    print(f"  AUC for {class_name}: {class_auc:.4f}")

Inference: Model 1/5


Processing rows: 100%|██████████| 28762/28762 [00:15<00:00, 1872.53it/s]


Inference: Model 2/5


Processing rows: 100%|██████████| 28762/28762 [00:15<00:00, 1895.38it/s]


Inference: Model 3/5


...

Inference: Model 4/5


Processing rows: 100%|██████████| 28762/28762 [00:14<00:00, 2006.62it/s]


Inference: Model 5/5


Processing rows: 100%|██████████| 28762/28762 [00:15<00:00, 1887.32it/s]



--- Improved Results ---
Ensemble Graphical Model AUC-ROC: 0.7086
  AUC for Fatal: 0.7700
  AUC for Minor/Other Injury: 0.6910
  AUC for Moderate Injury: 0.5970
  AUC for Non-casualty (towaway): 0.7852
  AUC for Serious Injury: 0.7512
